# EduMentor DPO Fine-Tuning — Phi-3.5-mini

**What this notebook does:**
1. Reads `dpo_pairs.jsonl` exported by your FastAPI app (upload it below)
2. Generates `chosen` responses using **Groq free tier** (your existing key)
3. Fine-tunes `microsoft/Phi-3.5-mini-instruct` with **DPO** using Unsloth on Colab T4
4. Exports a **Q4_K_M GGUF** — drop it into your server and `ollama create edumentor:latest`

**No paid APIs. No paid GPU. Everything free.**
- Groq: free tier, used only to generate chosen responses (one call per pair)
- Unsloth: free open-source library, 2x faster than HuggingFace TRL on T4
- Colab T4: free tier is enough for Phi-3.5-mini with 4-bit quantization
- Phi-3.5-mini: free on HuggingFace, same model as your Modelfile

**Before running:** Runtime → Change runtime type → T4 GPU

## Step 0 — Install dependencies

In [ ]:
# Unsloth install — one command, handles all CUDA-specific wheels automatically
!pip install unsloth --quiet
# TRL for DPO trainer, datasets for data loading, groq for chosen generation
!pip install trl datasets groq --quiet
print('All dependencies installed.')

## Step 1 — Upload your dpo_pairs.jsonl

In [ ]:
from google.colab import files
import json

print('Upload the dpo_pairs.jsonl file exported by your EduMentor app.')
print('It lives at:  data/dpo_pairs.jsonl  on your server.')
print('scp it to your laptop first, then upload here.')
uploaded = files.upload()

JSONL_FILENAME = list(uploaded.keys())[0]

raw_pairs = []
with open(JSONL_FILENAME, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            raw_pairs.append(json.loads(line))

print(f'Loaded {len(raw_pairs)} pairs.')
print('Sample prompt:', raw_pairs[0]['prompt'][:120] if raw_pairs else 'no data')

## Step 2 — Generate `chosen` responses via Groq (free tier)

In [ ]:
# Paste your Groq API key — the same one in your .env file
# Groq free tier: 30 requests/minute, 14400/day — more than enough
GROQ_API_KEY = 'gsk_...'   # <-- replace with your key

In [ ]:
import time
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

SYSTEM_PROMPT = """You are EduMentor, an expert Socratic AI tutor for school students.

Core rules:
- NEVER give direct answers — always guide with questions and hints
- Adapt to the student's level
- Acknowledge emotions before responding
- Be concise and clear
- Always respond in the same language the student used

You will be given a student question and a BAD previous response.
Write a BETTER response that follows the core rules above.
Return only the improved response text — no preamble, no explanation."""

def generate_chosen(prompt: str, rejected: str, topic: str, agent_type: str) -> str:
    """Call Groq to generate a better response for this prompt."""
    context = f'Topic: {topic}\n' if topic else ''
    user_msg = (
        f'{context}'
        f'Student asked: {prompt}\n\n'
        f'Previous bad response (do NOT copy this): {rejected}\n\n'
        f'Write a better Socratic response:'
    )
    response = client.chat.completions.create(
        model='meta-llama/llama-4-scout-17b-16e-instruct',  # same model as your quiz agent
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': user_msg},
        ],
        max_tokens=512,
        temperature=0.7,
    )
    return response.choices[0].message.content.strip()


# Generate chosen for all pairs that don't have one yet
complete_pairs = []
failed = 0

for i, pair in enumerate(raw_pairs):
    if pair.get('chosen'):          # already has a chosen (re-run safety)
        complete_pairs.append(pair)
        continue
    try:
        chosen = generate_chosen(
            pair['prompt'],
            pair['rejected'],
            pair.get('topic', ''),
            pair.get('agent_type', ''),
        )
        pair['chosen'] = chosen
        complete_pairs.append(pair)
        print(f'[{i+1}/{len(raw_pairs)}] ✓ chosen generated ({len(chosen)} chars)')
    except Exception as e:
        failed += 1
        print(f'[{i+1}/{len(raw_pairs)}] ✗ failed: {e}')
    # Groq free tier: 30 req/min — 2s gap keeps us well under the limit
    time.sleep(2)

print(f'\nDone. {len(complete_pairs)} complete pairs, {failed} failed.')

# Save with chosen filled in (so you can re-upload if Colab disconnects)
with open('dpo_pairs_complete.jsonl', 'w', encoding='utf-8') as f:
    for p in complete_pairs:
        f.write(json.dumps(p, ensure_ascii=False) + '\n')
print('Saved dpo_pairs_complete.jsonl')

## Step 3 — Load Phi-3.5-mini with Unsloth (4-bit, fits on T4)

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 2048   # matches your Modelfile num_ctx=4096 but 2048 is safe for T4

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='microsoft/Phi-3.5-mini-instruct',   # same base as your Modelfile GGUF
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,           # auto-detect (bfloat16 on T4)
    load_in_4bit=True,    # 4-bit quantization — keeps VRAM under 15 GB
)

# Attach LoRA adapters — only these layers are trained, not the full model
# r=16 and alpha=16 are standard starting values for DPO on small models
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,       # 0 is optimal for Unsloth
    bias='none',
    use_gradient_checkpointing='unsloth',   # saves VRAM
    random_state=42,
)
print('Model loaded with LoRA adapters.')

## Step 4 — Format dataset for DPO

In [ ]:
from datasets import Dataset

# Phi-3.5 chat template format
# Must match your Modelfile TEMPLATE exactly so the fine-tuned model
# behaves the same way when loaded back into Ollama
SYSTEM_MSG = (
    'You are EduMentor, a Socratic AI tutor.\n'
    'Core rules:\n'
    '- NEVER give direct answers — always guide with questions and hints\n'
    '- Adapt to the student level and explanation style\n'
    '- Acknowledge emotions before responding\n'
    '- Keep responses concise'
)

def format_phi35(system: str, user: str, assistant: str) -> str:
    """Format a single turn in Phi-3.5 chat template."""
    return (
        f'<|system|>\n{system}<|end|>\n'
        f'<|user|>\n{user}<|end|>\n'
        f'<|assistant|>\n{assistant}<|end|>'
    )

dpo_rows = []
for p in complete_pairs:
    if not p.get('chosen') or not p.get('rejected') or not p.get('prompt'):
        continue
    # DPO trainer expects: prompt, chosen, rejected — all as full formatted strings
    prompt_str   = format_phi35(SYSTEM_MSG, p['prompt'], '')  # no assistant part
    chosen_str   = format_phi35(SYSTEM_MSG, p['prompt'], p['chosen'])
    rejected_str = format_phi35(SYSTEM_MSG, p['prompt'], p['rejected'])
    dpo_rows.append({
        'prompt':   prompt_str,
        'chosen':   chosen_str,
        'rejected': rejected_str,
    })

dataset = Dataset.from_list(dpo_rows)
print(f'Dataset: {len(dataset)} rows')
print('Sample prompt (first 200 chars):', dataset[0]['prompt'][:200])

## Step 5 — DPO Training

In [ ]:
from trl import DPOTrainer, DPOConfig

# DPO config — conservative settings safe for T4 with a small dataset
# If you have < 20 pairs, num_train_epochs=3 is fine.
# If you have 50+ pairs, you can reduce to 1 epoch.
training_args = DPOConfig(
    output_dir='./dpo_output',
    num_train_epochs=3,
    per_device_train_batch_size=1,      # T4 has 15 GB; batch=1 is safe
    gradient_accumulation_steps=4,      # effective batch = 4
    learning_rate=5e-6,                 # lower than SFT — DPO is sensitive to LR
    beta=0.1,                           # DPO temperature: controls how far from reference
    fp16=True,                          # T4 does not support bfloat16
    logging_steps=1,
    save_strategy='no',                 # save only at end via GGUF export
    warmup_ratio=0.1,
    optim='adamw_8bit',                 # 8-bit optimizer saves VRAM
    report_to='none',                   # no wandb — keeps it simple
    max_length=MAX_SEQ_LEN,
    max_prompt_length=MAX_SEQ_LEN // 2,
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,     # Unsloth handles ref model internally — saves VRAM
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

trainer.train()
print('Training complete.')

## Step 6 — Export to GGUF (Q4_K_M — matches your Modelfile)

In [ ]:
# Save merged model as Q4_K_M GGUF — same quantization as your current model
# File will appear in the Colab file browser as edumentor_dpo.gguf
model.save_pretrained_gguf(
    'edumentor_dpo',
    tokenizer,
    quantization_method='q4_k_m',
)
print('GGUF saved as edumentor_dpo-unsloth.Q4_K_M.gguf')

## Step 7 — Download and deploy

1. Click the folder icon in Colab → right-click `edumentor_dpo-unsloth.Q4_K_M.gguf` → Download
2. On your server:
   ```bash
   # Copy the new GGUF into your models folder
   cp ~/Downloads/edumentor_dpo-unsloth.Q4_K_M.gguf ./models/phi-3.5-mini-instruct.Q4_K_M.gguf

   # Re-create the Ollama model (Modelfile stays unchanged)
   ollama create edumentor:latest -f Modelfile

   # Verify it loaded
   ollama list
   ```
3. Restart your FastAPI app — it picks up the new model automatically via Ollama.

**No config changes needed.** Your `Modelfile` already points to
`./models/phi-3.5-mini-instruct.Q4_K_M.gguf` so replacing the file and
re-running `ollama create` is all it takes.